<a href="https://www.kaggle.com/code/novcdt/lstm-dqn-vs-random-forest-beth-testing?scriptVersionId=321041273" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os, glob, pickle, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import deque

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Config ────────────────────────────────────────────────────────────────────
NEW_PACKETS   = 2_000          # evaluation set size
SEQ_LEN       = 10             # must match training
LSTM_MODEL    = '/kaggle/input/notebooks/novcdt/lstm-dqn3/lstm_dqn_keras_beth.h5'
LSTM_PREP     = '/kaggle/input/notebooks/novcdt/lstm-dqn3/preprocessing_keras.pkl'
RF_MODEL      = '/kaggle/input/notebooks/novcdt/random-forest/lstm_dqn_keras_beth.h5'
RF_PREP       = '/kaggle/input/notebooks/novcdt/random-forest/preprocessing_rf.pkl'

# ── Attack-type sets (identical to both training scripts) ─────────────────────
PROC_INJ_EVENTS  = {'clone', 'kill', 'memfd_create', 'mknod'}
C2_BEACON_EVENTS = {'connect', 'socket', 'bind', 'getsockname',
                    'accept', 'accept4', 'listen', 'dup', 'dup2', 'dup3'}

# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
POSSIBLE = [
    r"E:\MINI Project\IDS(Beth Dataset)\BETH dataset\labelled_testing_data.csv\labelled_testing_data.csv",
]
DATA_PATH = next((p for p in POSSIBLE if os.path.exists(p)), None)
if DATA_PATH is None:
    found = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    DATA_PATH = found[0] if found else None
assert DATA_PATH, "Dataset CSV not found. Update POSSIBLE paths above."

df_raw = pd.read_csv(DATA_PATH)
print(f"Full dataset shape : {df_raw.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  ATTACK-TYPE / SUBTYPE LABELLING  (same logic as both training scripts)
# ─────────────────────────────────────────────────────────────────────────────
def assign_attack_type(row):
    evt, sus, evil = row['eventName'], row['sus'], row['evil']
    if evil == 1:
        return 'Process_Injection' if evt in PROC_INJ_EVENTS else 'Privilege_Escalation'
    if sus == 1:
        return 'C2_Beaconing' if evt in C2_BEACON_EVENTS else 'Data_Exfiltration'
    return 'Normal'

df_raw['attack_type'] = df_raw.apply(assign_attack_type, axis=1)

df_s = df_raw.sort_values(['processId', 'timestamp']).copy()
df_s['prev_attack'] = df_s.groupby('processId')['attack_type'].shift(1)

def assign_subtype(row):
    if row['attack_type'] == 'Normal':
        return 'normal'
    return 'sequential' if row['prev_attack'] == row['attack_type'] else 'direct'

df_s['attack_subtype'] = df_s.apply(assign_subtype, axis=1)
df_s.drop(columns=['prev_attack'], inplace=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3.  STRATIFIED SAMPLING — NEW 2 000 PACKETS
#     We exclude the rows that were used during training by sampling from
#     the tail of each stratum (training used random_state=SEED; here we
#     use random_state=SEED+1 to get a fresh, non-overlapping set).
# ─────────────────────────────────────────────────────────────────────────────
df_s['strata'] = df_s['attack_type'] + '_' + df_s['attack_subtype']
strata_counts  = df_s['strata'].value_counts()
target_per_stratum = NEW_PACKETS // len(strata_counts)

sampled = (
    df_s.groupby('strata', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), target_per_stratum),
                                  random_state=SEED + 1))      # ← different seed
)
if len(sampled) < NEW_PACKETS:
    remaining = df_s[~df_s.index.isin(sampled.index)]
    extra = remaining.sample(
        min(NEW_PACKETS - len(sampled), len(remaining)), random_state=SEED + 1
    )
    sampled = pd.concat([sampled, extra])

sampled = sampled.sort_values(['processId', 'timestamp']).reset_index(drop=True)
print(f"New evaluation sample : {sampled.shape}")
print(sampled['attack_type'].value_counts(), "\n")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  FEATURE ENGINEERING  (identical column set for both models)
# ─────────────────────────────────────────────────────────────────────────────
NUMERIC_COLS = ['timestamp', 'processId', 'parentProcessId', 'userId',
                'eventId', 'argsNum', 'returnValue']
FEATURE_COLS = NUMERIC_COLS + ['eventName_enc', 'processName_enc',
                               'sus', 'evil', 'subtype_enc']

subtype_map = {'normal': 0, 'direct': 1, 'sequential': 2}

def engineer_features(df, le_event, le_process, le_label, scaler):
    """Apply saved encoders/scaler to a new dataframe."""
    df = df.copy()

    # Handle unseen categories gracefully
    def safe_transform(le, values):
        known = set(le.classes_)
        arr = np.array([v if v in known else le.classes_[0] for v in values])
        return le.transform(arr)

    df['eventName_enc']   = safe_transform(le_event,   df['eventName'])
    df['processName_enc'] = safe_transform(le_process, df['processName'])
    df['subtype_enc']     = df['attack_subtype'].map(subtype_map).fillna(0).astype(int)
    df['label']           = safe_transform(le_label,   df['attack_type'])

    X = df[FEATURE_COLS].values.astype(np.float32)
    X = scaler.transform(X)
    y = df['label'].values
    return X, y, df

# ─────────────────────────────────────────────────────────────────────────────
# 5.  LOAD MODELS & PREPROCESSING OBJECTS
# ─────────────────────────────────────────────────────────────────────────────
print("Loading LSTM-DQN model …")
lstm_model = tf.keras.models.load_model(LSTM_MODEL, compile=False)

with open(LSTM_PREP, 'rb') as f:
    lstm_prep = pickle.load(f)

print("Loading Random Forest model …")
with open(RF_MODEL, 'rb') as f:
    rf_model = pickle.load(f)

with open(RF_PREP, 'rb') as f:
    rf_prep = pickle.load(f)

# Class names (use LSTM's label encoder — both are identical)
class_names = list(lstm_prep['le_label'].classes_)
print(f"Classes : {class_names}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  PREPARE FEATURES
# ─────────────────────────────────────────────────────────────────────────────
# ── For Random Forest (flat, no sequences) ───────────────────────────────────
X_flat, y_true, df_feat = engineer_features(
    sampled,
    rf_prep['le_event'], rf_prep['le_process'],
    rf_prep['le_label'], rf_prep['scaler']
)

# ── For LSTM-DQN (sequences of length SEQ_LEN) ───────────────────────────────
def build_sequences(X, y, process_ids, seq_len):
    sequences, labels, indices = [], [], []
    for pid in np.unique(process_ids):
        mask  = process_ids == pid
        x_pid = X[mask]
        y_pid = y[mask]
        orig  = np.where(mask)[0]
        for i in range(seq_len - 1, len(x_pid)):
            sequences.append(x_pid[i - seq_len + 1 : i + 1])
            labels.append(y_pid[i])
            indices.append(orig[i])
    return (np.array(sequences, dtype=np.float32),
            np.array(labels,    dtype=np.int64),
            np.array(indices,   dtype=np.int64))

process_ids_new = df_feat['processId'].values

# Re-apply LSTM scaler/encoders (may differ slightly from RF's)
X_lstm_flat, y_lstm_all, _ = engineer_features(
    sampled,
    lstm_prep['le_event'], lstm_prep['le_process'],
    lstm_prep['le_label'], lstm_prep['scaler']
)

X_seq, y_seq, seq_idx = build_sequences(
    X_lstm_flat, y_lstm_all, process_ids_new, SEQ_LEN
)
print(f"LSTM sequences    : {X_seq.shape}")
print(f"RF flat samples   : {X_flat.shape}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 7.  INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
# ── LSTM-DQN ─────────────────────────────────────────────────────────────────
print("Running LSTM-DQN inference …")
q_values  = lstm_model.predict(X_seq, batch_size=256, verbose=1)
lstm_preds = np.argmax(q_values, axis=1)
lstm_true  = y_seq

# ── Random Forest ─────────────────────────────────────────────────────────────
print("\nRunning Random Forest inference …")
rf_preds = rf_model.predict(X_flat)
rf_true  = y_true

# ─────────────────────────────────────────────────────────────────────────────
# 8.  METRICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_metrics(trues, preds, name, class_names):
    acc  = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        trues, preds, average='weighted', zero_division=0
    )
    report = classification_report(trues, preds,
                                   target_names=class_names,
                                   zero_division=0)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(report)
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    return acc, prec, rec, f1

lstm_acc, lstm_prec, lstm_rec, lstm_f1 = compute_metrics(
    lstm_true, lstm_preds, "LSTM-DQN (Keras)", class_names
)
rf_acc, rf_prec, rf_rec, rf_f1 = compute_metrics(
    rf_true, rf_preds, "Random Forest (Flat)", class_names
)

# ─────────────────────────────────────────────────────────────────────────────
# 9.  SAVE TEXT REPORT
# ─────────────────────────────────────────────────────────────────────────────
with open('model_comparison_report.txt', 'w') as fh:
    fh.write("MODEL COMPARISON REPORT — 2 000 New Packets\n")
    fh.write("="*60 + "\n\n")
    fh.write(f"{'Metric':<20} {'LSTM-DQN':>12} {'Random Forest':>14}\n")
    fh.write("-"*48 + "\n")
    for metric, v1, v2 in [
        ("Accuracy",  lstm_acc,  rf_acc),
        ("Precision", lstm_prec, rf_prec),
        ("Recall",    lstm_rec,  rf_rec),
        ("F1-Score",  lstm_f1,   rf_f1),
    ]:
        fh.write(f"{metric:<20} {v1:>12.4f} {v2:>14.4f}\n")
    fh.write("\n\n--- LSTM-DQN Classification Report ---\n")
    fh.write(classification_report(lstm_true, lstm_preds,
                                   target_names=class_names, zero_division=0))
    fh.write("\n\n--- Random Forest Classification Report ---\n")
    fh.write(classification_report(rf_true, rf_preds,
                                   target_names=class_names, zero_division=0))

print("\nReport saved → model_comparison_report.txt")

# ─────────────────────────────────────────────────────────────────────────────
# 10. PLOT 1 — SIDE-BY-SIDE CONFUSION MATRICES  (counts + normalised)
# ─────────────────────────────────────────────────────────────────────────────
def make_cms(trues, preds):
    cm      = confusion_matrix(trues, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    return cm, cm_norm

lstm_cm, lstm_cm_n = make_cms(lstm_true, lstm_preds)
rf_cm,   rf_cm_n   = make_cms(rf_true,   rf_preds)

fig = plt.figure(figsize=(20, 9))
fig.suptitle("Confusion Matrices — 2 000 New Packets",
             fontsize=15, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

kw_cnt  = dict(fmt='d',   cmap='Purples', linewidths=0.5)
kw_norm = dict(fmt='.2f', cmap='Blues',   linewidths=0.5, vmin=0, vmax=1)

def hm(ax, data, title, **kwargs):
    sns.heatmap(data, annot=True, ax=ax,
                xticklabels=class_names, yticklabels=class_names, **kwargs)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True',      fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

# Row 0 — LSTM-DQN
hm(fig.add_subplot(gs[0, :2]), lstm_cm,   "LSTM-DQN — Counts",      **kw_cnt)
hm(fig.add_subplot(gs[0, 2:]), lstm_cm_n, "LSTM-DQN — Normalised",  **kw_norm)

# Row 1 — Random Forest
hm(fig.add_subplot(gs[1, :2]), rf_cm,     "Random Forest — Counts",     **kw_cnt)
hm(fig.add_subplot(gs[1, 2:]), rf_cm_n,   "Random Forest — Normalised", **kw_norm)

plt.savefig('comparison_confusion_matrices.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved → comparison_confusion_matrices.png")

# ─────────────────────────────────────────────────────────────────────────────
# 11. PLOT 2 — METRIC BAR CHART COMPARISON
# ─────────────────────────────────────────────────────────────────────────────
metrics      = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
lstm_scores  = [lstm_acc,  lstm_prec,  lstm_rec,  lstm_f1]
rf_scores    = [rf_acc,    rf_prec,    rf_rec,    rf_f1]

x     = np.arange(len(metrics))
width = 0.32

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Model Comparison — Overall Metrics  (2 000 New Packets)",
             fontsize=13, fontweight='bold')

# ── Bar chart ─────────────────────────────────────────────────────────────────
ax = axes[0]
bars1 = ax.bar(x - width/2, lstm_scores, width, label='LSTM-DQN',
               color='steelblue',  edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, rf_scores,   width, label='Random Forest',
               color='darkorange', edgecolor='white', linewidth=0.8)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=10)
ax.set_title('Overall Metric Comparison', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ── Per-class F1 grouped bar chart ────────────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support as prfs

_, _, lstm_f1_pc, lstm_sup = prfs(lstm_true, lstm_preds, zero_division=0)
_, _, rf_f1_pc,   rf_sup   = prfs(rf_true,   rf_preds,   zero_division=0)

# align class lists (both encoders should be identical, but be safe)
n_cls = len(class_names)
x2    = np.arange(n_cls)

ax2 = axes[1]
b1 = ax2.bar(x2 - width/2, lstm_f1_pc[:n_cls], width, label='LSTM-DQN',
             color='steelblue',  edgecolor='white', linewidth=0.8)
b2 = ax2.bar(x2 + width/2, rf_f1_pc[:n_cls],   width, label='Random Forest',
             color='darkorange', edgecolor='white', linewidth=0.8)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
             f'{h:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax2.set_xticks(x2)
ax2.set_xticklabels(class_names, rotation=22, ha='right', fontsize=8.5)
ax2.set_ylim(0, 1.15)
ax2.set_ylabel('F1-Score', fontsize=10)
ax2.set_title('Per-Class F1-Score Comparison', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, axis='y', alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('comparison_metrics_bar.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved → comparison_metrics_bar.png")

# ─────────────────────────────────────────────────────────────────────────────
# 12. FINAL SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL COMPARISON SUMMARY — 2 000 New Packets")
print("="*60)
print(f"{'Metric':<20} {'LSTM-DQN':>12} {'Random Forest':>14} {'Winner':>10}")
print("-"*58)
rows = [
    ("Accuracy",  lstm_acc,  rf_acc),
    ("Precision", lstm_prec, rf_prec),
    ("Recall",    lstm_rec,  rf_rec),
    ("F1-Score",  lstm_f1,   rf_f1),
]
for name, v1, v2 in rows:
    winner = "LSTM-DQN" if v1 >= v2 else "Rand Forest"
    print(f"{name:<20} {v1:>12.4f} {v2:>14.4f} {winner:>10}")
print("="*60)
print("\nOutputs:")
print("  comparison_confusion_matrices.png")
print("  comparison_metrics_bar.png")
print("  model_comparison_report.txt")